In [1]:
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

from catboost import CatBoostRegressor

In [2]:
train_sample_path = "../train_sample.csv"
test_sample_path = "../test_sample.csv"

In [3]:
train_sample = pd.read_csv(train_sample_path)
train_sample.head(2)

,start_point,end_point,time_of_day,day_of_week,traffic_condition,event_count,is_holiday,vehicle_density,population_density,weather,public_transport_availability,historical_delay_factor,travel_time
0,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),day,Sunday,NaN,9,1,NaN,high,NaN,1,0.878909,26.907612
1,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),morning,Thursday,NaN,7,1,medium,high,NaN,1,1.081668,27.489129


In [4]:
test_sample = pd.read_csv(test_sample_path)
test_sample.head(2)

,start_point,end_point,time_of_day,day_of_week,traffic_condition,event_count,is_holiday,vehicle_density,population_density,weather,public_transport_availability,historical_delay_factor
0,West Jakarta (Jakarta Barat),East Jakarta (Jakarta Timur),morning,Saturday,5.0,8,1,medium,NaN,NaN,2,1.126429
1,South Jakarta (Jakarta Selatan),East Jakarta (Jakarta Timur),evening,Saturday,NaN,9,1,low,medium,fog,2,1.121015


In [5]:
time = train_sample["time_of_day"].str.split().str[0]
day = train_sample["day_of_week"].str.split().str[0]

train_sample["day_and_time"] = day + " " + time
test_sample["day_and_time"] = test_sample["day_of_week"].str.split().str[0] + " " + test_sample["time_of_day"].str.split().str[0]

In [6]:
start = train_sample["start_point"].str.split().str[0]
end = train_sample["end_point"].str.split().str[0]

train_sample["route"] = start + " " + end
test_sample["route"] = test_sample["start_point"].str.split().str[0] + " " + test_sample["end_point"].str.split().str[0]

In [7]:
route = train_sample["route"]
day_and_time = train_sample["day_and_time"]

train_sample["route_day_and_time"] = route + " " + day_and_time
test_sample["route_day_and_time"] = test_sample["route"] + " " + test_sample["day_and_time"]

In [8]:
route = train_sample["route"]

train_sample["route_public_transport"] = route + " " + train_sample["public_transport_availability"].astype(str)
test_sample["route_public_transport"] = test_sample["route"] + " " + test_sample["public_transport_availability"].astype(str)

In [21]:
train_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   start_point                    40000 non-null  str    
 1   end_point                      40000 non-null  str    
 2   time_of_day                    40000 non-null  str    
 3   day_of_week                    40000 non-null  str    
 4   traffic_condition              40000 non-null  float64
 5   event_count                    40000 non-null  int64  
 6   is_holiday                     40000 non-null  int64  
 7   vehicle_density                40000 non-null  str    
 8   population_density             40000 non-null  str    
 9   weather                        40000 non-null  str    
 10  public_transport_availability  40000 non-null  int64  
 11  historical_delay_factor        40000 non-null  float64
 12  travel_time                    40000 non-null  float64
 1

In [9]:
reference_cols = [
    'start_point',
    'end_point',
    'time_of_day',
    'day_of_week',
    'event_count',
    'is_holiday',
    'public_transport_availability',
    'route'
]

target_cols = [
    'traffic_condition',
    'vehicle_density',
    'population_density',
    'weather'
]

target_cols = [
    'traffic_condition',
    'vehicle_density',
    'population_density',
    'weather'
]

# Urutan level fallback
reference_levels = [
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week',
        'event_count',
        'is_holiday',
        'public_transport_availability'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week',
        'is_holiday',
        'public_transport_availability'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day'
    ],
    [
        'start_point',
        'end_point'
    ]
]

for target in target_cols:

    for refs in reference_levels:

        # Hanya proses yang masih missing
        mask = train_sample[target].isna()

        if not mask.any():
            break

        # Cari modus berdasarkan grup
        mode_map = (
            train_sample.dropna(subset=[target])
              .groupby(refs)[target]
              .agg(lambda x: x.mode().iloc[0])
        )

        # Mapping ke baris yang masih missing
        filled = (
            train_sample.loc[mask, refs]
              .merge(
                  mode_map.rename('mode_value'),
                  left_on=refs,
                  right_index=True,
                  how='left'
              )['mode_value']
        )

        # Isi hanya yang berhasil mendapatkan modus
        train_sample.loc[mask, target] = filled.values

In [10]:
train_sample.isnull().sum()

start_point                      0
end_point                        0
time_of_day                      0
day_of_week                      0
traffic_condition                0
event_count                      0
is_holiday                       0
vehicle_density                  0
population_density               0
weather                          0
public_transport_availability    0
historical_delay_factor          0
travel_time                      0
day_and_time                     0
route                            0
route_day_and_time               0
route_public_transport           0
dtype: int64

In [11]:
test_sample.isnull().sum()

start_point                        0
end_point                          0
time_of_day                        0
day_of_week                        0
traffic_condition                600
event_count                        0
is_holiday                         0
vehicle_density                  600
population_density               600
weather                          600
public_transport_availability      0
historical_delay_factor            0
day_and_time                       0
route                              0
route_day_and_time                 0
route_public_transport             0
dtype: int64

In [12]:
reference_cols = [
    'start_point',
    'end_point',
    'time_of_day',
    'day_of_week',
    'event_count',
    'is_holiday',
    'public_transport_availability',
    'route'
]

target_cols = [
    'traffic_condition',
    'vehicle_density',
    'population_density',
    'weather'
]

# Urutan level fallback
reference_levels = [
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week',
        'event_count',
        'is_holiday',
        'public_transport_availability'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week',
        'is_holiday',
        'public_transport_availability'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day'
    ],
    [
        'start_point',
        'end_point'
    ]
]

for target in target_cols:

    for refs in reference_levels:

        # Hanya proses yang masih missing
        mask = test_sample[target].isna()

        if not mask.any():
            break

        # Cari modus berdasarkan grup
        mode_map = (
            test_sample.dropna(subset=[target])
              .groupby(refs)[target]
              .agg(lambda x: x.mode().iloc[0])
        )

        # Mapping ke baris yang masih missing
        filled = (
            test_sample.loc[mask, refs]
              .merge(
                  mode_map.rename('mode_value'),
                  left_on=refs,
                  right_index=True,
                  how='left'
              )['mode_value']
        )

        # Isi hanya yang berhasil mendapatkan modus
        test_sample.loc[mask, target] = filled.values

In [13]:
test_sample.isnull().sum()

start_point                      0
end_point                        0
time_of_day                      0
day_of_week                      0
traffic_condition                0
event_count                      0
is_holiday                       0
vehicle_density                  0
population_density               0
weather                          0
public_transport_availability    0
historical_delay_factor          0
day_and_time                     0
route                            0
route_day_and_time               0
route_public_transport           0
dtype: int64

In [14]:
X_test = test_sample

In [15]:
y_train = train_sample['travel_time']
X_train = train_sample[[
    "start_point",
    "end_point",
    "time_of_day",
    "day_of_week",
    "vehicle_density",
    "population_density",
    "weather",
    "is_holiday",
    "route",
    "public_transport_availability",
    "day_and_time",
    "route_day_and_time",
    "route_public_transport"
    ]]
X_test = test_sample[[
    "start_point",
    "end_point",
    "time_of_day",
    "day_of_week",
    "vehicle_density",
    "population_density",
    "weather",
    "is_holiday",
    "route",
    "public_transport_availability",
    "day_and_time",
    "route_day_and_time",
    "route_public_transport"
    ]]
# etc.
# your code here

In [16]:
cat_cols = [
    "start_point",
    "end_point",
    "time_of_day",
    "day_of_week",
    "vehicle_density",
    "population_density",
    "weather",
    "is_holiday",
    "route",
    "public_transport_availability",
    "day_and_time",
    "route_day_and_time",
    "route_public_transport"
]

model = CatBoostRegressor(
    iterations=300,
    depth=3,
    loss_function="RMSE",
    cat_features=tuple(cat_cols),
    verbose=100,
    random_seed=38
)

In [17]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
    return_train_score=True
)

# print("CV RMSE:", -scores["test_score"].mean())

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

Learning rate set to 0.188406
0:	learn: 13.0293377	total: 213ms	remaining: 1m 3s
100:	learn: 4.7653519	total: 4.93s	remaining: 9.71s
200:	learn: 4.7059059	total: 9.85s	remaining: 4.85s
299:	learn: 4.6738518	total: 14.6s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0149950	total: 72.5ms	remaining: 21.7s
100:	learn: 4.8411325	total: 4.54s	remaining: 8.95s
200:	learn: 4.7546666	total: 9.11s	remaining: 4.49s
299:	learn: 4.6971214	total: 13.8s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 12.9002765	total: 47.8ms	remaining: 14.3s
100:	learn: 4.5782512	total: 4.54s	remaining: 8.95s
200:	learn: 4.5217431	total: 9.35s	remaining: 4.6s
299:	learn: 4.4895492	total: 14.1s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0340327	total: 48.8ms	remaining: 14.6s
100:	learn: 4.7684115	total: 4.6s	remaining: 9.07s
200:	learn: 4.6987679	total: 9.28s	remaining: 4.57s
299:	learn: 4.6705326	total: 14.2s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0025555	total: 3

In [18]:
model.fit(X_train, y_train)

Learning rate set to 0.195167
0:	learn: 12.9019594	total: 55.3ms	remaining: 16.5s
100:	learn: 4.6810390	total: 5.21s	remaining: 10.3s
200:	learn: 4.6372095	total: 11s	remaining: 5.4s
299:	learn: 4.6080242	total: 15.9s	remaining: 0us


CatBoostRegressor(cat_features=('start_point', 'end_point', 'time_of_day', 'day_of_week', 'vehicle_density', 'population_density', 'weather', 'is_holiday', 'route', 'public_transport_availability', 'day_and_time', 'route_day_and_time', 'route_public_transport'), depth=3, iterations=300, loss_function='RMSE', random_seed=38, verbose=100)

In [19]:
y_train_hat = model.predict(X_train)
mse_lr = mean_squared_error(y_train_hat, y_train)
r2_lr = r2_score(y_train_hat, y_train)
mse_lr, r2_lr

(20.912793278904935, 0.8947422930343032)

In [20]:
y_hat_test = model.predict(X_test)
pd.DataFrame(y_hat_test).to_csv('submission.csv', index=False)